# Reeling in Smart Attackers' Phishing Emails — Colab notebook

**CSE 587 — Group 5**: Parker Davis · Dillon VanGilder · Emmanuel Adjei Domfeh.

End-to-end pipeline:
1. Install dependencies
2. Clone the project from GitHub
3. (Optional) Mount Google Drive to back up checkpoints
4. Download the Champa et al. curated phishing dataset
5. Train BERT baseline
6. Train **with smart-attacker augmentation**
7. Compare the two stages + per-edit ablation

Runtime: **A100 GPU** (Runtime → Change runtime type → A100 GPU). Falls back to T4/V100 automatically if A100 isn't available — the train script uses bfloat16 on A100/H100 and fp16+GradScaler on older GPUs.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 \
    scikit-learn pandas matplotlib seaborn nltk requests pyarrow

## 2. Clone the project from GitHub

The repo is public, so HTTPS clone works without auth.

In [ ]:
!git clone https://github.com/eadomfeh1/cse587-phishing.git
%cd cse587-phishing

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
print('CWD:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

## 2b. (Optional) Back up results to Google Drive

Colab runtimes disconnect after a while and lose `/content`. Mounting Drive lets us copy `results/` somewhere persistent at the end of each run. Skip this cell if you'd rather just download the JSON manually.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/cse587_results', exist_ok=True)

## 3. Download the dataset (Champa et al. 2024)

In [ ]:
!python scripts/download_data.py
!ls -lh data/raw

## 4. Inspect the dataset

In [ ]:
from src.data_loader import load_raw_csvs
df = load_raw_csvs()
print('Total rows:', len(df))
print(df.label.value_counts())
df.head(3)

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt
sns.countplot(x='label', data=df)
plt.title('Class balance (0 = benign, 1 = phishing)')
plt.show()

## 5. Train baseline BERT (A100, batch 64, bfloat16)

In [ ]:
# A100: batch 64 fits comfortably in 40GB VRAM with bfloat16 autocast.
# If you're on T4 (16GB), reduce to --batch_size 16.
!python -m src.train --model bert-base-uncased --epochs 3 --batch_size 64

In [ ]:
# Back up baseline results to Drive (skip if you didn't mount Drive)
!cp -r results /content/drive/MyDrive/cse587_results/baseline_$(date +%Y%m%d_%H%M%S) 2>/dev/null && echo 'baseline saved to Drive' || echo 'Drive not mounted — skipping backup'

## 6. Train with smart-attacker augmentation

In [ ]:
!python -m src.train --model bert-base-uncased --epochs 3 --augment --augment_factor 1 --batch_size 64

In [ ]:
!cp -r results /content/drive/MyDrive/cse587_results/augmented_$(date +%Y%m%d_%H%M%S) 2>/dev/null && echo 'augmented saved to Drive' || echo 'Drive not mounted — skipping backup'

## 7. Compare reports

In [ ]:
import json
from pathlib import Path
ra = json.loads(Path('results/eval_report.json').read_text())
ra

In [ ]:
import matplotlib.pyplot as plt
labels = ['accuracy', 'f1_phish', 'recall_phish', 'macro_f1']
std = [ra['standard'][k] for k in labels]
sm  = [ra['smart_attacker'][k] for k in labels]
x = range(len(labels))
fig, ax = plt.subplots(figsize=(7, 4))
w = 0.35
ax.bar([i - w/2 for i in x], std, w, label='Standard test')
ax.bar([i + w/2 for i in x], sm,  w, label='Smart-attacker test')
ax.set_xticks(list(x)); ax.set_xticklabels(labels, rotation=20)
ax.set_ylim(0, 1)
ax.legend()
ax.set_title('Standard vs smart-attacker test (last run)')
plt.tight_layout(); plt.show()

## 8. Per-edit ablation

How much does each individual smart-attacker edit hurt the model?

In [ ]:
abl = ra['ablation']
ops = list(abl.keys())
f1s = [abl[o]['f1_phish'] for o in ops]
fig, ax = plt.subplots(figsize=(6,4))
ax.bar(ops, f1s, color='steelblue')
ax.axhline(ra['standard']['f1_phish'], color='black', linestyle='--', label='standard F1')
ax.set_ylim(0, 1); ax.set_ylabel('F1 (phishing)')
ax.set_title('F1 drop when each smart-attacker edit is applied alone')
plt.xticks(rotation=20); plt.legend(); plt.tight_layout(); plt.show()

## 9. Inspect a few smart-attacker examples

In [ ]:
from src.augmentation import AugmentationConfig, make_smart_attacker_eval
from src.data_loader import load_splits
splits = load_splits()
smart = make_smart_attacker_eval(splits['test'].head(20).copy(), AugmentationConfig())
for _, row in smart[smart.label == 1].head(3).iterrows():
    print('---')
    print(row['text'][:600])